<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/rag/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# import os

# repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
# repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

# if os.path.exists("../"+repo_name):
#     print("Repository already present, update...")
#     !git pull
# else:
#     print("Repository clone...")
#     !git clone -b rag {repo_url}
#     %cd {repo_name}

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
# parquet_path = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection.parquet'

In [20]:
# import importlib
# import rag
# # from rag import load_parquet
# importlib.reload(rag)

# ds = rag.load_parquet(parquet_patth)

In [21]:
# ds['content']

In [22]:
# import phase
!pip install beir rank_bm25 faiss-cpu hnswlib ir_measures

from rank_bm25 import BM25Okapi
import numpy as np
import faiss
import hnswlib
import time
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import torch
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from datasets import load_dataset


In [23]:
import os
parquet_path = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection.parquet'
if (os.path.exists(parquet_path)):
  print("Ok")

ds = load_dataset("parquet", data_files=parquet_path, split="train")

corpus_docs = list(ds['content'])
corpus_ids = list(range(len(corpus_docs)))
# corpus_docs[:6]




Ok


In [24]:

bi_enc = SentenceTransformer('BAAI/bge-m3', model_kwargs={"torch_dtype": torch.float16},)
#x_enc = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def embed_sparse(docs):
    return list(map(lambda text: text.split(" "), docs))

def embed_dense(encoder, docs):
    return encoder.encode(docs, convert_to_numpy=True, batch_size=4, show_progress_bar=True, normalize_embeddings=True)

def dense_search(query, top_k):
    # get similarity scores
    query_embedding = embed_dense(bi_enc, [query])[:1]
    doc_inds, scores = index_hnsw.knn_query(query_embedding, k=top_k)

    # format to run
    run = {corpus_ids[int(ind)]: float(s) for ind, s in zip(doc_inds[0], scores[0])}
    return run

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [25]:
# pip install -U bitsandbytes accelerate transformers

In [27]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline

# model_id = "meta-llama/Llama-3.1-8B-Instruct"

# # 1. Configurazione della Quantizzazione 4-bit (il "salvavita" per i tuoi 12GB)
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,     # Risparmia ulteriore memoria
#     bnb_4bit_quant_type="nf4",          # Migliore precisione per modelli Llama
#     bnb_4bit_compute_dtype=torch.bfloat16 # Velocizza il calcolo su GPU moderne
# )

# # 2. Caricamento Tokenizer e Modello
# tokenizer = AutoTokenizer.from_pretrained(model_id)
# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     quantization_config=bnb_config,
#     device_map="auto",                  # Distribuisce il modello in modo intelligente
#     torch_dtype=torch.bfloat16,
#     low_cpu_mem_usage=True              # Ottimizza l'uso della RAM di sistema in fase di caricamento
# )

# # 3. Creazione del componente generativo (Pipeline)
# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=512,                 # Limita l'output per non saturare la RAM
#     temperature=0.7,
#     do_sample=True
# )

# # Test rapido
# # prompt = "Explaining Kurdish history in 3 sentences:"
# # print(pipe(prompt)[0]['generated_text'])

In [28]:
# create dense index processing dataset in chunks and storing them in the drive at the specified path
#doc_embeddings = embed_dense(bi_enc, corpus_docs)

output_path="/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings"

import pickle

chunk_size = 5000  # Numero di documenti per ogni salvataggio
start_index = 0    # Cambia questo valore se devi ripartire da un punto interrotto

for i in range(start_index, len(corpus_docs), chunk_size):
    end_index = min(i + chunk_size, len(corpus_docs))
    print(f"Processando blocchi da {i} a {end_index}...")

    # Prendi la fetta di documenti
    batch_docs = corpus_docs[i:end_index]

    # Genera gli embedding (usa la tua funzione embed_dense con batch_size=4)
    embeddings = embed_dense(bi_enc, batch_docs)

    # Salva il pezzetto su Drive
    file_name = f"embeddings_chunk_{i}_{end_index}.pkl"
    with open(os.path.join(output_path, file_name), 'wb') as f:
        pickle.dump(embeddings, f)

    print(f"Salvato: {file_name}")

In [29]:
#recover and combine the saved embedded chunks

import os
import pickle
import numpy as np
import re

path="/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings"

# 1. Recupera tutti i file .pkl
files = [f for f in os.listdir(path) if f.endswith('.pkl')]

# 2. Funzione per estrarre il primo numero dal nome file per l'ordinamento corretto
def extract_number(filename):
    match = re.search(r'chunk_(\d+)_', filename)
    return int(match.group(1)) if match else 0

files.sort(key=extract_number)

# 3. Carica e concatena
all_chunks = []
for file in files:
    with open(os.path.join(path, file), 'rb') as f:
        chunk = pickle.load(f)
        all_chunks.append(chunk)
    print(f"Caricato: {file}")

# Trasforma la lista in un unico array NumPy
doc_embeddings = np.vstack(all_chunks)

print(f"Forma finale dei doc_embeddings: {doc_embeddings.shape}")

Caricato: embeddings_chunk_0_5000.pkl
Caricato: embeddings_chunk_5000_10000.pkl
Caricato: embeddings_chunk_10000_15000.pkl
Caricato: embeddings_chunk_15000_20000.pkl
Caricato: embeddings_chunk_20000_25000.pkl
Caricato: embeddings_chunk_25000_30000.pkl
Caricato: embeddings_chunk_30000_35000.pkl
Caricato: embeddings_chunk_35000_40000.pkl
Caricato: embeddings_chunk_40000_45000.pkl
Caricato: embeddings_chunk_45000_50000.pkl
Caricato: embeddings_chunk_50000_55000.pkl
Caricato: embeddings_chunk_55000_60000.pkl
Caricato: embeddings_chunk_60000_65000.pkl
Caricato: embeddings_chunk_65000_70000.pkl
Caricato: embeddings_chunk_70000_75000.pkl
Caricato: embeddings_chunk_75000_80000.pkl
Caricato: embeddings_chunk_80000_85000.pkl
Caricato: embeddings_chunk_85000_90000.pkl
Caricato: embeddings_chunk_90000_95000.pkl
Caricato: embeddings_chunk_95000_100000.pkl
Caricato: embeddings_chunk_100000_105000.pkl
Caricato: embeddings_chunk_105000_110000.pkl
Caricato: embeddings_chunk_110000_115000.pkl
Caricato: 

In [30]:
dim = doc_embeddings.shape[1]
index_hnsw = hnswlib.Index(space='cosine', dim=dim)
index_hnsw.init_index(max_elements=len(corpus_docs), ef_construction=200, M=32)
index_hnsw.add_items(doc_embeddings, list(range(len(corpus_ids))))
print("Finito")

# create generative component
pipe = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

Finito


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [47]:
#Save complete embeddings at the specified path
index_path = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embeddings/kurdish_wiki_index.bin"
index_hnsw.save_index(index_path)
print("Indice salvato! Ora puoi respirare.")

Indice salvato! Ora puoi respirare.


In [34]:

def rag(query, top_k):
    # create system and user messages
    system_prompt = "You are helpful assistant. Answer the question given the supportive information."
    user_prompt = f"Question: {query}"

    # search relevant documents
    if top_k > 0:
        # runs = rrf_search(query, top_k)
        runs = dense_search(query, top_k)
        docs = [corpus_docs[doc_id] for doc_id in runs]
        docs_context = "\n".join(docs)
        user_prompt += f"\n Referenced knowledge: {docs_context}"

    # format prompt
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # generate the answer
    outputs = pipe(
        prompt,
        max_new_tokens=128,
        do_sample=True,
        temperature=1.1,
        eos_token_id=pipe.tokenizer.eos_token_id,
        pad_token_id=pipe.tokenizer.pad_token_id
    )
    predicted_answer = outputs[0]['generated_text'][len(prompt):].strip()

    return prompt, predicted_answer, docs

In [32]:
# top_k = 0
# prompt, predicted_answer = rag("How many letters does the Tajik alphabet consist of?", top_k)
# print(f"Prompt: {prompt}\nPredicted answer: {predicted_answer}")

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'eos_token_id', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


UnboundLocalError: cannot access local variable 'docs' where it is not associated with a value

In [45]:
top_k = 10
prompt, predicted_answer, docs = rag("Di elfubêya tacîkî de çend tîp hene?", top_k)
i=0
for doc in docs:
  print(f"Doc {i}:{doc}\n")
  i=i+1
print("==================Answer=============")
print(f"Prompt: {prompt}\nPredicted answer: {predicted_answer}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Doc 0:Elfubêy tacîkî elfubêyekî Kirîlî ye ke bo nûsînî zimanî tacîkî kelkî lêwerdegîrê. Em elfubêye le 35 pît pêkhatûw e.

Doc 1:Di nav bakteriyofajan de herî zêde li ser bakteriyofaja T4ê de xebatên zanistî hatiye kirin. Loma li vir bakteriyofaja T4 wekî mînak tê nîşankirin. Faj T4 tûşê bakteriya ya Escherichia coli dibe. Faj T4 taybetmendiyên vîrusên lûlpêçî (bi inglîzî: helical) û firerûyî (bi înglîzî: polyhedral) lixwe digirin, loma ji van vîrusan re tê gotin vîrusên aloz. Serê van vîrusên aloz dişibe vîrusên bîstrûyî (bi înglîzî: icosahedral), kilikên wan jî dişibe vîrusên lûlpêçî.
Faj T4 ji sê beşên serekî; ji serî, stû û kilikê pêk tê. Di baktriyofaja T4ê de genoma vîrusê, ango ADN di nav serê firerûyî de cih digire. Kilik bi kalanê kilikê ve pêçayî ye. Kalanê kilikî girjok e û bi şeweyî lûlpêçî ye. Li kotahiya kilikê de binik cih digire. Rîşalên kilikê bi binikê ve girêdayî ne. Di faj T4ê de şeş heb rîşalên kilîkê hene û ji bo naskirina xaneya xanexwê wekî wergir kar dikin.

Do

In [46]:
#Debug: test single query vs document
# 1. Prendi il blocco "incriminato" e la tua domanda
target_doc = """Elfubêy tacîkî elfubêyekî Kirîlî ye ke bo nûsînî zimanî tacîkî kelkî lêwerdegîrê. Em elfubêye le 35 pît pêkhatûw e."""
query = "Di elfubêya tacîkî de çend tîp hene?"

# 2. Genera gli embedding solo per questi due
target_emb = bi_enc.encode(["passage: " + target_doc], normalize_embeddings=True)
query_emb = bi_enc.encode([query], normalize_embeddings=True)

# 3. Calcola la similarità coseno manuale (deve essere alta, es. > 0.7)
similarity = np.dot(query_emb, target_emb.T)
print(f"Similarità tra domanda e blocco specifico: {similarity[0][0]:.4f}")

Similarità tra domanda e blocco specifico: 0.6050
